# Example 9.5 / 2: Use the Built-in tab Handler

Companion notebook for **9.5 Choosing a Table Handler** in the AMPL lab Documentation, section "Use the Built-in `tab` Handler". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Use the Built-in `tab` Handler

The built-in `tab` and `bit` handlers are always loaded. A declaration with no handler string
uses `tab` and the file `TABLE_NAME.tab`, which makes it handy for saving values and for testing
a mapping without any external driver:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param rate {PROD} > 0;

table Rates: PROD <-> [product], rate;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}

In [5]:
# run.py
ampl.write_table("Rates")
print(open("Rates.tab").read())
ampl.eval("reset data;")
ampl.read_table("Rates")
print(ampl.param["rate"].to_dict())

ampl.tab 1 1
product	rate
bands	200
coils	140
plate	160

{'bands': 200, 'coils': 140, 'plate': 160}


### Pure AMPL: data.dat and model.run

The first line of a `.tab` file is a header that AMPL writes for itself; the rest is one row per
key. `PROD <-> [product]` writes the set's members and reads them back after `reset data;`. The
`.tab` format is not a substitute for a documented business exchange format.

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
write table Rates;
reset data;
read table Rates;
display rate;

rate [*] :=
bands  200
coils  140
plate  160
;

